<a href="https://colab.research.google.com/github/adnanzaidi548/PhD-_Thesis_Data_Codes-/blob/main/MASTER_PAPER01_NEO_AMS_STANDALONE_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MASTER_PAPER01_NEO_AMS — Standalone v2
Upload **only this notebook** to Jupyter or Colab if desired. Support scripts are embedded and restored automatically. Research results remain labelled as synthetic where applicable.


## 1. Environment and pinned source identifiers
All downloaded source metadata and outputs are logged. GPU is not required. Dependencies: numpy, pandas, scipy, matplotlib, nbformat, requests; optional gdown. Reference environment information is captured. Editing the constants below changes run identity.

In [1]:
from pathlib import Path
import os, sys, json, csv, re, shutil, subprocess, hashlib, time, platform
import numpy as np, pandas as pd
from scipy import stats
from datetime import datetime, timezone
RUN_TAG=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
HERE=Path.cwd().resolve()
ROOT=HERE if (HERE/'support').exists() else (Path('/content/Paper01_MASTER_Notebook_Execution_Package') if Path('/content/Paper01_MASTER_Notebook_Execution_Package/support').exists() else HERE)
OUT=ROOT/'outputs'/RUN_TAG; OUT.mkdir(parents=True,exist_ok=True)
SEED_RANGE=list(range(100,130)); N_BOOTSTRAP=2000
REPO='adnanzaidi548/PhD-_Thesis_Data_Codes-'
GITHUB_URL='https://github.com/'+REPO
DRIVE_FOLDERS={'real':'1Y8Mr7vEkLxkor0sFM6eYKI0GoN4iAvAH','processed':'1qlQbS2IC2C7F8eYileZ5TBVHhNgooj5e','baseline':'1-i1M0bU92aqsg7iUD61m9n1O7cG2taAE','processing_code':'16O19P53CpiN1f95brBtBCynGpkqz_Hmo','processed_bundles':'1gaM4MEG7xKYZuyakWW8hOcLzu_omegba'}
print('Root:',ROOT,'Output:',OUT,'Python:',sys.version.split()[0]); print('Source:',GITHUB_URL)
# Standalone fallback: restore bundled research scripts when only this .ipynb was uploaded.
# The compressed source is embedded in the notebook; no untrusted remote code is executed.
import base64, zlib
EMBEDDED_SUPPORT = {'leakage_free_reanalysis.py': 'eNqtGWtv27b2u38F0X2QlNBqkvbi3jrhBXq7bClumw5rtovBMARaomM6eoGkknhD//s9h6RetrOm2Aw0lajD836SL168eF9mohbwpzQkF/xuulJCECV4yfOtlppUK6K3pVkLI1OyFGW6Lri6i8l1RdJGKdg3XVeNIjw1Dc81kSVZVUqkXJt48osWmqzko8iIFvCn4CZdy/KWkkxqI8vUkBoAp1WZb7ttmhJeZoSTVZPnU1nCOuyTVUnE47SutJl8fP/hJ9LUtVA9RyQsKwObQJy82vJlLkhalUZVeS5UFJO3eU6aUhpNuBK9SPHkxYsXE1nUlTIk1feUbHRVUgAANowsxGSlqgK4NOtcLokH/Ale201lU9RbwjUpawerU1lv46qG3fJ30W4pZF5T8kGWgqt3VamN4rI0lPynaspMT37+9OmGIdowSVYyF0kSxUroKr8XYRTXHBU9AaZi5CSWpRbKhCcU0IS41UO8DE7+mbyrMpFcV0Ysq+pOB1Hk2FJNmfTa8lwBIsPLVIDAsmhybuAJthlkr05Sieu3ZbLKZZ3Ukyt29vr8mp2eTSaZWJFaiUymJkQk1NAC6EazCYEf6PQHb00CPpRWRd0YdAKjYAPaGjmqliDFvTWtJg/SrK3CgaGiJhfExGgZxCZXxDB2MlPCNKokr+MTu4yew5D2PADZg8V8ZhYtPPLCWAAeAj5sQGoRtNtXecVNiJvn09NFtLMD/QUc9NUOeFnHBcSE31bIMnxFTTRbRG4/2FIL8ivPG3GpVKVCqwunJfD0Gt3eqamucplura4GxCjPCmnYjWpaDWZVKRhQ/V2oSofXNDPbWjAwTR6dpxAD7CQ+T7laViU+yfKe5zJjJ+dK1FwqeGi0yPCTEhuRWhUzpzYIJ4KGB6bLWxFeeYL4czHMnMhDzZpFdI7WZodM3m+/5zJn840lselJXEeoYAxPlGq+Wdj4bvFD1tECaGwWF8xcuNVM8Aw045ZdNsjz0O6uFxZ9jegdMPIULOJbYcINnYNJFh1HQNYpHCwrslUwsyzGGlw/vBNblvNimXGymYUH6NJN1Msm8hGuUqjb7VfR1dWDUA7XAfxPYrd5C2zxjezu0hzh12K256SOYA+1gYBk88V5laZNLcHaGlTaf+6sarnqvQZ/BQRi6zIFxxzvZRxBYZhZPwAkLZEZUo15jUUo3EQd8ZhnWVhEh03p1e8YHvgbLiCkY8RwJVcr674X8SsCQAfUNjUX7Oywx/R2GEnxHfmk5K2EAtmVrCl/wKqCUaw1VqpUSSOUxEpiE1tZEc0L4Uol1ll+K+IR1idl0U2x4053LgTuhlB3F2wTXdj4PHbQt0pmScrrYPE8yWtVQXHleWLJsH2ym8WYubFtRrv/7TLJHifHp2L6ZqxNn7qO2emuq9icOAbG38MayqOTGxPDM/j8Bm46a7CCP4a4mw4C7+5A4N2BHumufaLo3Lq1EkUFBRzc2uXlXSltXLZh12fqIVj9TZYYLc98urV1pQOBjC4esD9iWMZq6pQTnaNeWD3tPvdpXWOzQfD7BTukw391oFh0jlmHwlaqY4bQR8OiMgjMyFcxDxW/dvXU1V5barBHkKlIFDYKvjahYK4gRxFFIonrRSFs8Y0KH4c6ubtNqzPmaNCOsQQZTe4e1gwfqPfBpHVizfwKdWZLuK+g7pU+rCvA4RaTQYHtH33th/5jBSvJoIlNsBG0euh7pUvX1xJImxn0W2tDbPfYFDPIFY8yrW6hG1vLFIrgFnroR9dYesUQaGRLaC9Lt4ryu8xy+QgcEtDUVOeQc3W6FlmT2+7bp2hoqNO80fJemi3Fdi4VmbDdIBpjCga2IdapjbjE69DfrKGzkxiEjlts6NOBCNj0cTBapShxrXnKG1DuoC2PWwVMXF5dQlJVW/I431CzoBZQlk3VaMvPfGetY2vu+75f2fXR1fHZ0dU5Sfve6VcXHel8Bl8XbHoqXj/RB6Vz3G8WbM9LnVNWDzZO8wr/riX8da0aGDrtOvoQPUTTHEOV2hll2GDtsdVyckfvkRe7ecYhhRyz+z5kgXJbInkEDLQvlkoEvLQLjqDrabPHrmGgZrZB0SajHNF2Zz1/AzHmISCAhspE9DTaV9aCnsD6SIQnmkqfzcEFwme3fGCLJzhBuidjun/SCo65+AqTzgchcYJ2uEHIos4lDLCIFUJDa0sM/ToXOM0sBdZ/YvawPMn8sXut6V1Ep16tdz1HBsSbgodAumiF/BOe8VPRf3r1tNION+T73RpjxY5dvyO25liJYM61UzhjNhzJcR+B59jgLLmBnmf7FWca1oFhr7rPI2rLBiSqitpH/zJwgnw5DKnzxr5CzvNvGNEzu3OxV7r85+Or2WI4RrrgKY2ApJtDYkSEKpm3aGFHRDuSkGz8uKGYTewpHW4dPNOlHfSZm/fDfEmbJdauVkea7Z4N4NDJNVeKb0PMABEkFbqWEa1qV27+CHBeTnLIvdCjnp7QoIAxHcIruQX5ZvEJ/E6/dBOuih8xX1+DGLNhgUW+E5i6TaNtF69i9xJR+wUmcg3dKlOxf6Ljioz4dmowLkV7hfwv0ukGceyHQBaXz/GQZM1r8Bd6FUXRy+sdVtyudovL70d7+X0/zODnSvgDdvJJqu9DPHmh1hAuBm1jb49jKsi9YfAQUAgHzGEsCCI8EVr1wfrAAEX8Pejhf3Y0CFd0JUWelTAVaJZD8bQ2np9g7/gQW6prSIkA2b0jgPOEyQTsmSS4OUlgUkmSAjwmSfykAoW8UpktVkZoo9tKhVJWFY7eZP4DtDmCYme4GKcUe1DXaeL0BPLBq5PdfF5q1h4dhbiBAloG/6K9hOsPPGymRLqhHcTp6GgGorud6Wh/KIKr3Qj29PoI0WI/pWdseAJDB/xgV24V1VZP32/aydfLhI6KctF78BZeGub2HwdJcIw46NFRFkU786Ef8Ox5I0ELzMAkdc5T4L89NX25aiA0xPgQLOUltkrpGnUPDtnOl4fL2BxMQ/9BT8/o2asDghdVJlc4xrcnNtF5u+RTHdrQP8ZpVW/DPYi5mUEn8ubNmz3stvpUmTPqyAgDUy0OT1d+ouBLHQ4OlNrzpGm71vLSfoguYNg4PTuI0vr5c+xomKGtWhmipTWyk7FTb8Y+4PFg9WXgjZngwXiC/RlMDqopdQwgAfUedHira3gh+Sc4T6D/QDZznNrN9tFthQxVQO97KFLByjshCrjwC54LiSz8Q80D752By2T2q+fsy07oKsXmag/K1od5ACTBKzCQXeM/QMwYPC72HLEQzhWDUbIGHxilYXjfGclgZW/qQs8ZzVuH/Ocex7O5mgPhobAK+/IHKI+3azY8wA4RPtrD4tU9chifwrpIh/8pUFEyZfAfLVkOWd6iozh6svHJsP0ABV0mwAdDXuDRsoN/nuNbLU/WNfxL5E+60eue5xt/q0F29M8Z+hrGFHzZM8HzPalL30mXKb7sedYSZgQspk4aKBrJOMm4gtFjoD3WIeABF1r+TWK0HH45UHMGvdscx7npcnRs50OXR4t937Rd3tiJs30o5xMHHXjgtEqsBBpcHNI5bQXoJPFOnjkPTyD9ttv9uUt/6NL6+cDNKd4VuasiNrg2AnTfkFoH6dGtuK22RYRer8n/ejtz+vx25nxI9znl5ejoz858nlJEd2piIZ30Q8peB7yUK1CPK+g5X4qcBZ9/u765urx5/458uv7wW0Arfzad6FTJ2rBgdO8X1xD67ZeR+vsrXwcjsMyhztjcKu3szaKrm0lX2Fhgr27XEG1AOOX5gUs9kA3ykMYrvYCOJWVBWZVTh40+ed/bXvXi5OT1AlxA23x2QlsZXJ+LzuuKKu1ueFvHclsQoq3Z1N7Uekau5I9XnwNqhynHPZsH2IyBYnKUocHLS3i0OdJCQMqBsRc4UCX4WKsc2xXl8E3j+YEga9EodzbWRqA/Diure5ETO+vB9v5UDLb2p2j2qtu3hUvhb7cR3Oos3zoe5OAWv16DEdESIoMesrN84DON97je3EnrVjHefAeRmzVAoY8mxJU4g8Ffhy0UlZYMO/OuXCv0/eCnt+9/vvwe9DfUb/Dh8u1/3/54SW4uP998dh+9eQIUzK2MHB3vGS3Cz798/Pj259+C/hjE9TyuLs6Gdw+QlV3Kg/YHQMKvlL5o5kioAwKQd58+3+zSdA70FElbzob4D6J34g5Fnc9egUn+D6NJ92w=', 'run_benchmark.py': 'eNqtWnuP2zYS/38/BZHiTlKWVmxvtpfYywLtpb0UaDaHJoegMASBlmibu7KkivI6bpDvfjND6mU73fRwATYrUcPhPH6cB7lPnjy5/fHt6Ps379i7Wq4Vm85YpcqqSHeJXmaKvfvt9v3rH9///E+m83Rn6krLjJlko9JdpvM1W6o82WxldR9e3BYwVWajfVFlaZ98q6TZVWqr8towmacsL1hdSZ2rlP36yzN4zM2qqLaqYktpFPBVJrx48uTJhd6WRVWzfLctD0walpecJeaBsztT5JxtpNlkenmxqootK2WNL8zN+Te82g+mlrU2tU5M8w0Eyi9+ffv2vUAqP45XOlNxHISVMkX2oPwgLGUF4l68FtPnF7diMr14I64uLlK1AsWAYZ4o3yiV8qJIxU8yM4qvK53GiSyBmOeFNkpMwnEwu2Dwr8rXIi9DUDQttiFwkbusjmGUmAREs5XJBjRHOgmEa+XfBn97Y6erDExIn8A5StY+EVXy4C/G/Dl/wSfTKOBXllGqZIo2FG7aZUf8gr/kk3H09LmlLIu9qgSIEe5yjR7wr8Ip/za84reOoFKpSpQxRWXEJz1b6NFVxICSabADs2Ii9Wcir+ue9K8DJ3uOg1v5UW93Wx/Evbx6CgNG575f16PrAN9K/WwyDS5RlBzkkBkQgvmYXgFYagZWZgqMzKbhC/46sJzhG4zPYIGnYhxeX1sRZKVXK1xxv1GVwiW+E5Pr4O/wcCOmk4CPw6sX8N/keSNfvatyluqk9hsXuN+8sbv7zVvLNg/cWpD+52gs0bcYR93hhzuh7K8OKM2Dgwv9T4CCH47AEBYdBDsD5stkrXzEH6yb6eQgvKTIYfsktcfBKSqRphYe8jl4XKZbXYv31U5x8G4dF/f08igikf/Cw5W96HIynlgzJcW2zFRNMtV+MGdpYbH6h6oK49/ytD6USui8hm9JAXKMwzlTW22MLnJDb2CKeGeABb4ku6qWOqNnnT/ITMM40pRSoyHheZdvleUDmL/HBxIF1CWWi4heEY11h8bXTkFHuZOZsBrB6l60qKM5oRrcDYt88gqwXqa8mSXlXqkqA8ECohoM9icCgP16NOHjIOLOxDMcG3M7dYhdO5M86kVPpxAIgs+LxkVRK+A3sFlrVW11TgGK3RVLiJAPYBeJwRdVAzOPTAZ7IC/yEUiutmWtHxQoUSDMIAZ36uJEsbijeXedSW6DZh+h0xZ3EQXhRjmCNih4F92I+saONvimYaKWWebT7NLu/xLZW2I0pxeFa8DFHV9EQdDXb+LcBS6smhA3Zz+++okBE7Cs2qOmI7mHeIvqg166qHSt/5A4reUECjjMC0+lK3AYKhvCNqv9e3UQmdwuU8nuZv4ZDfidCxn4D/JLn1euqvXhUXa0wy2vM/wH3I36a7IdL9BjBj4uC9wwC0BtkSS7UjcbsKVpfU2LdtinlIL70S3qbG+XGFCBObaEDmDSLDJrlg5lWao89e+CefMtlGnqb4PzrnHmHCzwDTMbvarBNioh7FqY7+WBUYLe6PUGoK0TxTbFDuImW+4Iqx6kagggbFetIRm3ic0MuHdG6gG/GUThrP42+FIEuIEUgPA7440RJonorGptqD1WrtnWDsSwHUYYeynw9UqlWRN3mDUSk6n6fSeTw1/TxkASHULm3u7I+2PK+xtxF9yINtpd2mlN1vGir7MBeE9tZY7hCZTayjrZMA3uA3dAuaUqyEnZAfjcKVyErQCFUOz1TYjYcilo1gXfp2IyHo+/QGdjDmawmUsOl2Iy7+0HEKtSuCTbVwXUofcfnr35AFJANF3JpFveJuhTo91Fp+YNhpEUA5Nwwf3YdEOEAMvv2imXEzV6OcRIp8LxxrMqDkbP5QUjV6o+uOTIlvi0LR6w/sboXTcbhPbVDlyVsRWMYwF/wnq/gVq3wwna+Sut86cqtuyh7qkFJsZmHn80BI7qM0GwdXZIuiqfOAfzpkA4NiYF3hORqIS4tAXaqJX/hKwPLDC+zhvzAQPoDcAVYCoogbaUOA0hNUEgI+KhL2mrI2Zbp0HqMqp6UGn8vyHx5NPM5XDYDu2qFJHvukmUVwWgx++v7QqVYI4wpq/HqOa2oOlPGhGvYMCbirhLQV/mro67FP1iyM1qJ2FBeEn17tOTaDxvi0RHMQ6fj+dU8RGM4IFK5qBf1iF5k5qodMfEIaAudlaKkwL2gchU7pPXA64+qmRXD7601rYl+f2e1uckO771Ve+3CVSa98tkKtq7VAtvHrRjLnPYX1193jy4Ah2n0VPQluk45J6Drls4cZbtGgaVJscIEHe1OmrZvgXkWMiycYW11dHHZ7ccvRSDP0FvkBvfeOub+H6dFFPRvvO2diM0gLk2ooFG71u75YhglRWyMRJV1SFuhyDoM4Pg5Wq/lt8zxAGGHP4lBgF3MMQzBlrKvXMXemPra2jUhRvhNpDETTfRxBXwgsO/TdPEjcIIRyjGDVTwxbVmy6KoDdQGZZxo/6Pt3Kbj6bf8B+g8x80ZgKTe2NhO/KPrl0ghCGp/fj4wpxMLQ300PPjYaiSbAlzpS+7/wAnmZMYyg+Rn+zwOHbcRE4yYBN2FNR7wgKoDc4TyiSv/PQgozPyOYcYPx9NrHr78xzWU8a7xXOfxKtNlXLbKTUE10OxrVANACrk0vrSiB48rm++yjBjCrL6qC+i+JlFP3+CpDCxXp2qrq1XVn1z6yOw7AUIEDirP/B8ugdLqtodOQ8WJefDx9IhXxd44lfa63rCixCiBX7y9xwGi1Px7XoAxf9Vlm70AFuEriA0fkGHlr/hKqyzN5VYZkUEO95H1Yox5bR/SqhvIgEDZviMBUYFoUBjEMU6OYyg843grdR7HrvB8gNAJ/oP+16dOaNizcnsYZRFwloBOBNrvtl5vOtrhx+5w4bHProt+bPpQti+wJyLXQ6I9+n0+ngMBSBcdo2g2SJWIoK7xhQKTT67GwXE1ZsTpIR78DNsi6nKLjAK3jdUUoJF744LTegMSfO+chp+ZDxhdeI6BF4mV9wmIPsefkOxz7F36HtF6bXVoD768rhDxAiCKlzJFdl6/cHakHqwBhmvyY2X16rCOJ5/PPBvu4griXQijnkU/kcJO2YKEqrU9RD9oz+DVG6QQ8NwgZ8D7UcKAkbOxHcZPIjPR9mMyDJwJxt4pHMDLRzhwBsav2Iar1MeuuW95IqsIK6j3EUYq0LUakqChK5fcBcKFaueBM4V7jE6AZA2IrJwpT6HzgMlpUS0sQV8+E82zgm+0GCQapA9OuLSuG1RHzaGik6+Rk9u1hP3FcyoJiC/HsCoqKJVSyhJu9AUkWv3yOs6KvfsIgjWD2Ma70Y2G0eA87qyEh7ivjAVgK7ud+A2zEGW1MrVh2ExDSGRXY9rlZu7OLFKFemL7Q4cH9tzaniEuvgYqj7hGik9VU+fNznnnK6HRhruYgl1sd/nnE6TgDQhJirE77kVuN4O7mD1gw4+596IFP7tyGz+iUyQu/08qoyqfT7ineiW6Gwm5uI9Gy8EZhtuwMoiCc7iH+aewtw4/i/khxiu1Uhir1BccwrEilhDdiorkd5sCF6U9EcNTw8FWGTDQlDbd5gC5uz0BuwGLKFtDiX49hWyDPw3PVhxtiiZK23EXp1VudK0fdH3oYx1LVILQCz4BvE/55PkR5qldsPshvOYTPuVXRzj42mR6mvR6eXVw09G/5YAyqCf7wG1fnNPdiBz1MdVxVoqok8HxYXaKmm4AP52mn+hLAasnqY1T3UATqV4NjmukAYEohXEKXu40BrJffaB9AoWvpPPzppKwZTRNo6P28c0ZtW7E5ChrHU+DOWcz5g10Ty/ObmCLetisk3PMhu0kSnB7ToKtzPUK9BTkQej3VCa87ub43c9v/vPL9+9/fnvL3t7+8pvH1ygbbTEPyo+4u0MuoYQoD/UGmj+PLn098npMCBQLguD0ZeQ2QUx5QFyNsYCLK7XWWG5jMwyekFm/9mApLogHZmh+3JUKioQCMEtOAWhgMBQephDO7j/gzwaCgayWBexWiBJFBRUNxmNqCMfhc24S6A+EN5nSHQ3OpAM4zq6aiw54LqHZVymGC/bq+39xtzftNTSgXCaw+pyZQ15vFELHHozYa/JOerMrywyMkWnwlnTXXx7mP3ve0r9tT2UNcctW1e01e6V0Do5L6BIeDw2rHI3RBbuW/OQq3n4pNwejE2Cv0jVas8yKA/KCr7D5y0xDw5P2/zDA3fMvZQ0742Cz0EpbS9AxLVYUINcKLwVgospB88Kmdk6pHf+YoK9YWZQYZLB6jIa4g52SVLqsY7OR0+tvAc7uLwNCO+C7nXwCNrzxl2m8PMAuhfgdbtTHVK+Bo7tgcfPaOXGzYoh/gADTbayo1cfax5EwBdAav6HimgoTMQ2a63Q82vF+3WFti2mFthD32sLHjnZ1EHdpwI670M/7EcnN6EWkoM0DhrJpw2zWP7M2beoe2yPghWcTJBTGeArwSJ0fNJO6cppmfalkOZ9qz1UywcwaCSLLfwEacohW', 'check_contract.py': 'eNplU8GOmzAQvecrrFwMkhs1Ui+bLIeozSGHVO2m1R5WK8sxA5nG2AgbWlr13zuG7C6onOD5eZj33sxyuTyixUoZBr9At0GdDbDT/rj7/O3wkWlnQ6N0YPoC+uoFQ2NaT1DADpjvbbhAQM0q8F6V4Jmzpl8tl8vFw/7r98PD/hPL2B8eVFNC4BteKI0GQy+NUzkXvLUY4esjvb+eYU5QbZQNckf4xTX421l54Zt1vKGhCQotEUNfQ6Q6+iJiroKSHTQenSW4WxNWofdoS0v9EWSdBQJVWTZQkoaBd3FtY3r+d5FDQYKrmg7IhOSmKd0sGD0NhLaxTBnzcrAiSck1zbKOFa5hV9ExtOxF9woDVD5JU6ZszqZXOLXUggxYUS8FGREhW3Ci3s+JOWiMYubcdyN5gQWT0qoKpMwyLmVFpkjJx3Y7ZTDPctQheelIvP03uxOz2tn6fTpcU96Tu1MXhkLj4aASekFQC1HrUzIGKPjxkacimUUouKPhaAb8lr/gdYMaBugtVfEhfv+XK+WJNGl4boecImcWsOA+KDMWm6YcR+Rq3c/xyjTqOCFISUd8msF6nT6PrsXnrPJsEL3Sru6TdEvAE8l+zgbd25tF1s1sIhKVje7Et6EYabUU15fd6bRhd+TXD9ABcuahotGOS4O+UoE2y9N0jYfbMTjmVVXTHiqtoSZ4xQ7TvXvdymHbeLr4B4EJRWQ='}
support_dir=ROOT/'support'
support_dir.mkdir(parents=True, exist_ok=True)
for filename, payload in EMBEDDED_SUPPORT.items():
    dest=support_dir/filename
    if not dest.exists():
        dest.write_bytes(zlib.decompress(base64.b64decode(payload)))
        print('Restored:', dest)
assert (support_dir/'leakage_free_reanalysis.py').exists(), 'Support recovery failed.'


Root: /content Output: /content/outputs/20261011T143632Z Python: 3.13.16
Source: https://github.com/adnanzaidi548/PhD-_Thesis_Data_Codes-
Restored: /content/support/leakage_free_reanalysis.py
Restored: /content/support/run_benchmark.py
Restored: /content/support/check_contract.py


## 2. External sources: public repository and optional Google Drive mount
GitHub API is only used to inspect/download selected plain-text data. Full repository checkout is optional. For Google Drive, Colab **mounts your authenticated Drive account** and locates files locally; a shared folder link is not necessarily present in `MyDrive`. Add a shortcut to your Drive, or supply your local folder path. Avoid automatic downloads of multi-hundred-MB archives and never expose credentials.

In [2]:
import requests
S=requests.Session(); S.headers.update({'Accept':'application/vnd.github+json','User-Agent':'Paper01-Repro-Notebook'})
repo_meta=None
try:
    response=S.get(f'https://api.github.com/repos/{REPO}',timeout=20); response.raise_for_status(); repo_meta=response.json()
    sha_response=S.get(f'https://api.github.com/repos/{REPO}/git/trees/{repo_meta["default_branch"]}?recursive=1',timeout=30);sha_response.raise_for_status();tree=sha_response.json()
    inventory=pd.DataFrame([{'path':x['path'],'size':x.get('size'),'git_sha':x.get('sha'),'type':x.get('type')} for x in tree.get('tree',[])])
    inventory.to_csv(OUT/'github_inventory.csv',index=False)
    assert not tree.get('truncated',False),'GitHub API tree truncated: inventory incomplete'
    SOURCE_SHA=tree.get('sha'); print('GitHub tree SHA:',SOURCE_SHA,'records:',len(inventory))
except Exception as exc:
    SOURCE_SHA=None; inventory=pd.DataFrame(); print('SOURCE FETCH UNAVAILABLE:',repr(exc))

GitHub tree SHA: 7fbe79f8e40a44ca3256046a2aac303eb95c7714 records: 75


In [3]:
# Optional in Colab. Never mount implicitly unless instructed.
MOUNT_DRIVE=False
if MOUNT_DRIVE and 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
# Set to a directory with real CSVs, or leave None. Filename alone never certifies manufacturing labels.
REAL_DATA_DIR=None  # e.g. '/content/drive/MyDrive/Paper01_Datasets'
PROCESSED_DATA_DIR=None
for name,folder_id in DRIVE_FOLDERS.items():
    print(name, 'https://drive.google.com/drive/folders/'+folder_id)

real https://drive.google.com/drive/folders/1Y8Mr7vEkLxkor0sFM6eYKI0GoN4iAvAH
processed https://drive.google.com/drive/folders/1qlQbS2IC2C7F8eYileZ5TBVHhNgooj5e
baseline https://drive.google.com/drive/folders/1-i1M0bU92aqsg7iUD61m9n1O7cG2taAE
processing_code https://drive.google.com/drive/folders/16O19P53CpiN1f95brBtBCynGpkqz_Hmo
processed_bundles https://drive.google.com/drive/folders/1gaM4MEG7xKYZuyakWW8hOcLzu_omegba


## 3. Download only explicit, text-based source artifacts
The notebook does not download all 75 GitHub files, ZIP archives or private Drive contents automatically. The whitelist avoids executing untrusted pickle objects and preserves raw bytes + hash for every downloaded source.

In [4]:
WHITELIST=['system_model_config.json','objective_definition.json','baseline_vs_neo_opt_comparison.csv','full_system_kpis.json','full_system_latency_summary.csv','robust_vs_deterministic_comparison.csv','reproducibility_manifest.json','01_dataset_manifest.json']
import urllib.parse
fetch_log=[]; raw_dir=OUT/'source_artifacts';raw_dir.mkdir(exist_ok=True)
for fname in WHITELIST:
    url='https://raw.githubusercontent.com/'+REPO+'/main/'+urllib.parse.quote(fname)
    try:
        r=S.get(url,timeout=30);r.raise_for_status();raw=r.content
        (raw_dir/fname).write_bytes(raw)
        fetch_log.append({'filename':fname,'url':url,'sha256':hashlib.sha256(raw).hexdigest(),'bytes':len(raw),'status':'fetched'})
    except Exception as exc:fetch_log.append({'filename':fname,'url':url,'status':'error:'+repr(exc)})
pd.DataFrame(fetch_log).to_csv(OUT/'source_fetch_log.csv',index=False)
print(pd.DataFrame(fetch_log)[['filename','status']].to_string(index=False))

                              filename  status
              system_model_config.json fetched
             objective_definition.json fetched
    baseline_vs_neo_opt_comparison.csv fetched
                 full_system_kpis.json fetched
       full_system_latency_summary.csv fetched
robust_vs_deterministic_comparison.csv fetched
         reproducibility_manifest.json fetched
              01_dataset_manifest.json fetched


## 4. Original-result consistency checks (no scale mixing)
Source values are **source-reported**, not results generated by this notebook. The following flags incompatible experiment definitions and documents observed service penalties.

In [5]:
audit=[]
try:
    a=pd.read_csv(raw_dir/'baseline_vs_neo_opt_comparison.csv').set_index('Metric')
    base=float(a.loc['Service Ratio','Baseline']); neo=float(a.loc['Service Ratio','NEO_Opt'])
    audit.append({'claim':'original_optimization_service','baseline':base,'proposed':neo,'delta':neo-base,'interpretation':'service worsens' if neo<base else 'no observed reduction','evidence_grade':'SOURCE_REPORTED'})
except Exception as e:audit.append({'claim':'original_optimization_service','interpretation':repr(e),'evidence_grade':'MISSING'})
try:
    f=json.loads((raw_dir/'full_system_kpis.json').read_text())
    audit.append({'claim':'full_system_top_level_latency','baseline':None,'proposed':f.get('latency_mean'),'delta':None,'interpretation':'Check measurement units and timing boundary before comparison','evidence_grade':'SOURCE_REPORTED'})
except Exception as e:audit.append({'claim':'full_system_top_level_latency','interpretation':repr(e),'evidence_grade':'MISSING'})
pd.DataFrame(audit).to_csv(OUT/'original_result_discrepancy_log.csv',index=False)
display(pd.DataFrame(audit))

,claim,baseline,proposed,delta,interpretation,evidence_grade
0,original_optimization_service,1.0,0.774186,-0.225814,service worsens,SOURCE_REPORTED
1,full_system_top_level_latency,NaN,30.289235,NaN,Check measurement units and timing boundary be...,SOURCE_REPORTED


## 5. Source data adapter: chronological forecasting without leakage
This optional module computes a **real-source time-series forecast benchmark** when the user supplies one local CSV with a verified timestamp and target. It does **not** magically create manufacturing jobs, machine capacities, tariffs, grid constraints, emissions measurements, or factory outcomes from energy series alone.

To configure, set `REAL_CSV`, `TIMESTAMP_COL`, `TARGET_COL`, and target units **after inspecting the source dataset documentation**. Chronological holdout is mandatory; rolling features access earlier observations only. No random time-series splitting.

In [6]:
REAL_CSV=None   # e.g. '/content/drive/MyDrive/Paper01_Datasets/site_energy.csv'
TIMESTAMP_COL=None  # e.g. 'timestamp'
TARGET_COL=None     # e.g. 'load_kw'
TARGET_UNITS=None   # e.g. 'kW'
HOLDOUT_FRACTION=.20
if REAL_CSV:
    assert TIMESTAMP_COL and TARGET_COL and TARGET_UNITS, 'Set explicit column and unit mappings before analysis'
    p=Path(REAL_CSV); assert p.exists(),f'Missing {p}'
    src=pd.read_csv(p)
    assert TIMESTAMP_COL in src and TARGET_COL in src,'Selected columns absent'
    x=src[[TIMESTAMP_COL,TARGET_COL]].copy(); x[TIMESTAMP_COL]=pd.to_datetime(x[TIMESTAMP_COL],utc=True,errors='raise');x[TARGET_COL]=pd.to_numeric(x[TARGET_COL],errors='coerce')
    assert not x[TIMESTAMP_COL].duplicated().any(),'Duplicate timestamps: resolve data provenance first'
    x=x.sort_values(TIMESTAMP_COL).dropna(); assert len(x)>=100,'Insufficient observations for a meaningful holdout'
    # One-step-ahead naive and trailing-window predictions require prior known actual only.
    x['persistence']=x[TARGET_COL].shift(1)
    x['rolling_3']=x[TARGET_COL].shift(1).rolling(3,min_periods=1).mean()
    split=int(len(x)*(1-HOLDOUT_FRACTION)); test=x.iloc[split:].dropna()
    assert len(test)>10
    actual=test[TARGET_COL].to_numpy();scores=[]
    for method in ['persistence','rolling_3']:
        forecast=test[method].to_numpy();scores.append({'model':method,'mae':float(np.abs(actual-forecast).mean()),'rmse':float(np.mean((actual-forecast)**2)**.5),'n_test':len(test),'target_units':TARGET_UNITS,'data_class':'PUBLIC_REAL_DATA__OR_USER_PROVIDED','source_csv_sha256':hashlib.sha256(p.read_bytes()).hexdigest()})
    pd.DataFrame(scores).to_csv(OUT/'real_series_forecast_scores.csv',index=False)
    print(pd.DataFrame(scores).to_string(index=False))
else:print('REAL-DATA FORECAST: NOT EXECUTED (set explicit file and mapping).')

REAL-DATA FORECAST: NOT EXECUTED (set explicit file and mapping).


## 6. Synthetic manufacturing scheduling: leakage-free repeated paired experiments
This experiment is **synthetic**, generates 12 one-slot jobs on 3 machines for 24 hours, and uses prior observations only to construct forecasts. `edf` and `energy` are heuristics; `contract` is a heuristic with forecast-aware gating. It does **not** establish optimality, real factory safety, or superiority to industrial controllers. Full-information MILP is explicitly hindsight (unavailable at decision time).

The reference script is copied into this package unmodified for provenance. It writes outputs beside its own file; this notebook archives them into this run folder.

In [7]:
# Execute the exact archived experiment script; capture the command, exit code and logs.
command=[sys.executable,str(ROOT/'support'/'leakage_free_reanalysis.py')]
proc=subprocess.run(command,cwd=str(ROOT),capture_output=True,text=True,timeout=300)
(OUT/'simulation_stdout.txt').write_text(proc.stdout)
(OUT/'simulation_stderr.txt').write_text(proc.stderr)
print('Exit:',proc.returncode,proc.stdout[-1400:],proc.stderr[-900:]);assert proc.returncode==0,'Benchmark execution failed; inspect logs'
for name in ['leakage_free_paired_runs.csv','leakage_free_summary.csv','leakage_free_paired_tests.csv','hindsight_milp.csv','causality_invariance_tests.csv']:
    src=ROOT/'support'/name
    if src.exists():shutil.copy2(src,OUT/name)
print('Generated:',[p.name for p in OUT.glob('leakage*')])

Exit: 0 8289212, 'ci_high': -0.02369333599903017, 'signflip_p': 0.04599080183963208}
{'ood': 1, 'metric': 'cost_currency', 'reference': 'contract_rolling3', 'baseline': 'edf_persistence', 'n': 30, 'mean_difference': -0.08612068473916222, 'ci_low': -0.1666550770556527, 'ci_high': -0.022890281070443722, 'signflip_p': 0.033193361327734455}
{'ood': 1, 'metric': 'cost_currency', 'reference': 'contract_rolling3', 'baseline': 'energy_rolling3', 'n': 30, 'mean_difference': -0.08612068473916216, 'ci_low': -0.16665507705565263, 'ci_high': -0.022890281070443573, 'signflip_p': 0.03239352129574085}
{'ood': 1, 'metric': 'cost_currency', 'reference': 'contract_rolling3', 'baseline': 'contract_persistence', 'n': 30, 'mean_difference': 0.02334401533782303, 'ci_low': -0.01139956501400725, 'ci_high': 0.06596768105425835, 'signflip_p': 0.5118976204759048}
MILP [{'seed': 100, 'ood': 0, 'milp_status': 0, 'milp_message': 'Optimization terminated successfully. (HiGHS Status 7: Optimal)', 'service_ratio': 1.0,

## 7. Scientific invariants and paired analysis
Service is not ignored; compare only identical seed/regime pairs. 95% CIs are paired bootstrap CIs, not confidence intervals over time steps. Each compared unit is one independently generated scheduling instance.

In [8]:
runs=pd.read_csv(OUT/'leakage_free_paired_runs.csv')
assert not runs.duplicated(['seed','ood','variant']).any()
assert runs['service_ratio'].between(0,1).all()
assert (runs['cost_currency']>=-1e-8).all()
assert (runs['emissions_kgco2']>=-1e-8).all()
summary=runs.groupby(['ood','variant'],as_index=False).agg(n=('seed','nunique'),cost_mean=('cost_currency','mean'),cost_sd=('cost_currency','std'),service_mean=('service_ratio','mean'),emissions_mean=('emissions_kgco2','mean'),repairs_mean=('repair_actions','mean'))
summary['evidence_grade']='SYNTHETIC_EXECUTED'; summary.to_csv(OUT/'master_summary.csv',index=False);display(summary)
comparisons=[]
for condition in sorted(runs['ood'].unique()):
    part=runs[runs['ood']==condition]
    pivot=part.pivot(index='seed',columns='variant',values='cost_currency')
    if not {'edf_persistence','contract_rolling3','contract_persistence'}.issubset(pivot.columns):continue
    for name in ['contract_rolling3','contract_persistence']:
        dif=(pivot[name]-pivot['edf_persistence']).dropna().to_numpy();rng=np.random.default_rng(321+int(condition));boot=np.mean(rng.choice(dif,(N_BOOTSTRAP,len(dif)),replace=True),axis=1)
        comparisons.append({'ood':int(condition),'policy':name,'control':'edf_persistence','n_pairs':len(dif),'mean_paired_cost_delta':float(dif.mean()),'ci95_low':float(np.quantile(boot,.025)),'ci95_high':float(np.quantile(boot,.975)),'cost_unit':'synthetic currency','evidence_grade':'SYNTHETIC_EXECUTED'})
pd.DataFrame(comparisons).to_csv(OUT/'paired_bootstrap_results.csv',index=False);display(pd.DataFrame(comparisons))

,ood,variant,n,cost_mean,cost_sd,service_mean,emissions_mean,repairs_mean,evidence_grade
0,0,contract_persistence,30,5.315340,0.421925,1.0,15.186686,0.666667,SYNTHETIC_EXECUTED
1,0,contract_rolling3,30,5.165809,0.466830,1.0,14.759455,0.566667,SYNTHETIC_EXECUTED
2,0,edf_persistence,30,5.391193,0.368636,1.0,15.403408,1.166667,SYNTHETIC_EXECUTED
3,0,energy_rolling3,30,5.391193,0.368636,1.0,15.403408,1.166667,SYNTHETIC_EXECUTED
4,1,contract_persistence,30,6.033353,0.374654,1.0,17.238150,1.100000,SYNTHETIC_EXECUTED
5,1,contract_rolling3,30,6.056697,0.350573,1.0,17.304848,0.900000,SYNTHETIC_EXECUTED
6,1,edf_persistence,30,6.142817,0.335461,1.0,17.550907,2.233333,SYNTHETIC_EXECUTED
7,1,energy_rolling3,30,6.142817,0.335461,1.0,17.550907,2.233333,SYNTHETIC_EXECUTED


,ood,policy,control,n_pairs,mean_paired_cost_delta,ci95_low,ci95_high,cost_unit,evidence_grade
0,0,contract_rolling3,edf_persistence,30,-0.225384,-0.368959,-0.107229,synthetic currency,SYNTHETIC_EXECUTED
1,0,contract_persistence,edf_persistence,30,-0.075853,-0.162889,0.000000,synthetic currency,SYNTHETIC_EXECUTED
2,1,contract_rolling3,edf_persistence,30,-0.086121,-0.165159,-0.025757,synthetic currency,SYNTHETIC_EXECUTED
3,1,contract_persistence,edf_persistence,30,-0.109465,-0.193013,-0.039977,synthetic currency,SYNTHETIC_EXECUTED


## 8. Contract feasibility and rejection/repair evidence
Rejecting invalid interface inputs and clipping/repairing a scalar control signal **does not prove** feasibility of a multi-machine manufacturing schedule. Existing proof obligations and limitations must be stated explicitly.

In [9]:
contract=ROOT/'support'/'check_contract.py'
if contract.exists():
    cp=subprocess.run([sys.executable,str(contract)],cwd=str(ROOT),capture_output=True,text=True,timeout=60)
    (OUT/'contract_test_stdout.txt').write_text(cp.stdout);(OUT/'contract_test_stderr.txt').write_text(cp.stderr)
    print('Contract script status:',cp.returncode,'output:',cp.stdout[-1100:]);assert cp.returncode==0
else:print('Contract script absent — no contract-test claim made')

Contract script status: 0 output: PASS: 9 injected semantic mismatches rejected; valid sample accepted. Illustrative contract only.



## 9. Figures and manifest
Plot only explicitly labelled synthetic outcomes. Machine-readable metadata provides source and generated file checksums and runtime versions.

In [10]:
import matplotlib.pyplot as plt
pivot=summary.pivot(index='variant',columns='ood',values='cost_mean')
fig,ax=plt.subplots(figsize=(8,4)); pivot.rename(columns={0:'nominal',1:'renewable stress'}).plot(kind='bar',ax=ax)
ax.set_ylabel('Synthetic cost (arbitrary currency)');ax.set_title('Synthetic matched-seed scheduling comparison; not factory evidence')
fig.tight_layout();fig.savefig(OUT/'synthetic_cost_by_regime.png',dpi=180);plt.close(fig)
manifest={'timestamp_utc':RUN_TAG,'github_repo':GITHUB_URL,'github_tree_sha':SOURCE_SHA,'drive_folder_ids':DRIVE_FOLDERS,'seeds':SEED_RANGE,'synthetic_scope':'3 machines, 12 one-slot jobs, 24 hours; generated renewable and tariffs','evidence_grade':'SYNTHETIC_EXECUTED','real_data_used':bool(REAL_CSV),'real_data_file':str(REAL_CSV) if REAL_CSV else None,'software':{'python':sys.version,'numpy':np.__version__,'pandas':pd.__version__},'files':{}}
for p in OUT.rglob('*'):
    if p.is_file() and p.name!='execution_manifest.json':manifest['files'][p.relative_to(OUT).as_posix()]=hashlib.sha256(p.read_bytes()).hexdigest()
(OUT/'execution_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Run files:',len(manifest['files']),'Output:',OUT)

Run files: 23 Output: /content/outputs/20261011T143632Z


## 10. Publication release gates (automatic FAIL-CLOSED)
Run completion is not publication readiness. A manuscript can claim quantified industrial benefits **only if** authentic manufacturing measurements, meaningful comparator parity, repeated trials, statistical tests, provenance, journal-relevant novelty, and reference checks have passed independently. Here the gates intentionally remain blocked until their evidence is supplied.

In [11]:
gates=[
('causal_synthetic_baselines',True,'This notebook reruns synthetic paired experiments'),
('real_source_series_forecast',bool(REAL_CSV),'Real time-series CSV and explicit unit mapping required'),
('manufacturing_ground_truth_jobs',False,'Measured plant jobs, deadlines, machine capacities and process telemetry not provided'),
('online_optimization_comparator',False,'Full-information MILP is hindsight; online optimizer unverified'),
('contract_semantic_advantage',False,'Feasibility reduction versus strong safety comparators not established'),
('multi_objective_service_parity',False,'Matched service constraints and cost/emissions comparison required'),
('original_notebook_pipeline_reexecution',False,'Google Drive origin notebooks/data packages must be executed separately'),
('bibliography_and_related_work_validation',False,'Publisher DOI checks and Paper04/Paper07 overlap clearance outstanding')]
gate_df=pd.DataFrame(gates,columns=['gate','passed','evidence_or_blocker']);gate_df.to_csv(OUT/'publication_release_gates.csv',index=False)
print('MANUSCRIPT SUBMISSION READY:',bool(gate_df.passed.all()));display(gate_df)

MANUSCRIPT SUBMISSION READY: False


,gate,passed,evidence_or_blocker
0,causal_synthetic_baselines,True,This notebook reruns synthetic paired experiments
1,real_source_series_forecast,False,Real time-series CSV and explicit unit mapping...
2,manufacturing_ground_truth_jobs,False,"Measured plant jobs, deadlines, machine capaci..."
3,online_optimization_comparator,False,Full-information MILP is hindsight; online opt...
4,contract_semantic_advantage,False,Feasibility reduction versus strong safety com...
5,multi_objective_service_parity,False,Matched service constraints and cost/emissions...
6,original_notebook_pipeline_reexecution,False,Google Drive origin notebooks/data packages mu...
7,bibliography_and_related_work_validation,False,Publisher DOI checks and Paper04/Paper07 overl...
